# Python Puzzles and Edge Cases — the Trap Questions

Screens use these to separate *users* of Python from *understanders*. Each puzzle below is executed, and the printed result **is** the answer — so before looking at the output, commit to a prediction. If your prediction is wrong, the explanation tells you the underlying rule, which is what you say when asked "why?" in the interview.

## 1. The Mutable Default Argument — the Most Famous Python Trap

**What does this print?** (Predict first.)

```python
def append_to(item, target=[]):
    target.append(item)
    return target
```

In [1]:
def append_to(item, target=[]):
    target.append(item)
    return target

print(append_to(1))
print(append_to(2))     # surprise?
print(append_to(3))

[1]
[1, 2]
[1, 2, 3]


**Why:** default values are evaluated **once, at function definition**, and stored on the function object. Every call without `target` shares the *same list*. The fix — and say this sentence: *"the default should be None, and the mutable object created inside the body"* —

```python
def append_to(item, target=None):
    if target is None:
        target = []
    target.append(item)
    return target
```

In [2]:
# proof the default lives on the function object
print("defaults dict:", append_to.__defaults__)
append_to.__defaults__[0].clear()        # mutate it from outside — same object

def append_fixed(item, target=None):
    if target is None:
        target = []
    target.append(item)
    return target
print(append_fixed(1), append_fixed(2))  # independent lists

defaults dict: ([1, 2, 3],)
[1] [2]


## 2. `is` vs `==` — and the Integer Caching Surprise

**Predict each line:**

In [3]:
a = [1, 2, 3]; b = [1, 2, 3]
print("lists  a == b :", a == b)
print("lists  a is b :", a is b)

x = 256; y = 256
print("ints   256 is 256:", x is y)

x = 257; y = 257
print("ints   257 is 257:", x is y)   # depends on interpreter — CPython caches -5..256

s1 = "hello"; s2 = "hello"
print("strs   'hello' is 'hello':", s1 is s2)
print("strs   'hello!' is 'hello!':", "hello!" is "hello!")

lists  a == b : True
lists  a is b : False
ints   256 is 256: True
ints   257 is 257: False
strs   'hello' is 'hello': True
strs   'hello!' is 'hello!': True


<>:13: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:13: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
/tmp/ipykernel_1350/3368578782.py:13: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  print("strs   'hello!' is 'hello!':", "hello!" is "hello!")


**The rule to state:** `==` calls `__eq__` (value equality); `is` compares object identity (same `id`). **Use `is` only for `None`, `True`, `False`, and sentinels.** The integer caching (−5 to 256) and string interning are CPython optimizations — never rely on them, which is exactly why `x is 257` behavior is "implementation detail," not language semantics.

## 3. Late Binding in Closures — the Loop Surprise

**What does this print?**

In [4]:
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])

[2, 2, 2]


**Why:** closures capture *variables*, not values — all three lambdas reference the same `i`, which is 2 after the loop ends. Two fixes — the default-argument trick or `functools.partial` — both of which bind the **current value**:

In [5]:
funcs_ok = [lambda i=i: i for i in range(3)]       # bind now via default arg
print([f() for f in funcs_ok])

from functools import partial
funcs_p = [partial(lambda v: v, i) for i in range(3)]
print([f() for f in funcs_p])

[0, 1, 2]
[0, 1, 2]


## 4. Chained Operations and the Boolean Trap

**Predict:** `False == False in [False]`? and `not True == False`?

In [6]:
print("False == False in [False] :", False == False in [False])
# chains as (False == False) and (False in [False]) -> True and True

print("not True == False          :", not True == False)
# == binds tighter than not -> not (True == False) -> True

print("True + True                 :", True + True)
print("sum([True, False, True])    :", sum([True, False, True]))

False == False in [False] : True
not True == False          : True
True + True                 : 2
sum([True, False, True])    : 2


**The rules:** comparisons chain left-to-right with an implicit `and`; arithmetic operators bind tighter than `not`/`and`/`or`; `bool` is a subclass of `int` (True = 1) — which is why `sum(list_of_bools)` counts Trues, a genuinely useful interview idiom.

## 5. Shallow vs Deep Copy — the Nested-List Trap

**Predict what changes:**

In [7]:
import copy
original = [[1, 2], [3, 4]]
shallow = original[:]              # the classic "I'll just slice it" copy
deep = copy.deepcopy(original)

shallow[0].append(99)
print("original after shallow[0].append(99):", original)
print("deep untouched                     :", deep)

original after shallow[0].append(99): [[1, 2, 99], [3, 4]]
deep untouched                     : [[1, 2], [3, 4]]


**The rule:** slicing (and `list(...)`, and `dict.copy()`) creates a new outer container holding **references to the same inner objects**. For nested structures you need `copy.deepcopy` — or, interview-favorite alternative, a comprehension that copies each row: `[[x for x in row] for row in m]`. Related pandas note: `df.copy()` is a deep copy; `df.copy(deep=False)` is the shallow one.

## 6. String Immutability and the Concatenation Trap

**Why is building a string with `+=` in a loop O(n²)?**

In [8]:
import timeit

def by_join(n):
    return ";".join(str(i) for i in range(n))

def by_plus(n):
    s = ""
    for i in range(n):
        s += str(i)
    return s

print("join    :", round(timeit.timeit(lambda: by_join(500), number=200), 3), "s")
print("+= loop :", round(timeit.timeit(lambda: by_plus(500), number=200), 3), "s")

join    : 0.009 s
+= loop : 0.009 s


**The answer:** strings are immutable, so each `+=` may allocate a fresh string and copy everything so far — quadratic total work. `''.join(parts)` allocates once. (CPython sometimes optimizes in-place `+=` when refcount is 1, but that's an implementation detail — the interview answer is "use join").

## 7. `dict.get` vs `[]`, `setdefault`, and `defaultdict` — Four Ways to Count

**Same task, four idioms — know all, recommend the last:**

In [9]:
from collections import defaultdict

words = ["ml", "sql", "ml", "dl", "sql", "ml"]

# 1) naive with KeyError risk
counts = {}
for w in words:
    if w not in counts:
        counts[w] = 0
    counts[w] += 1

# 2) get with default
counts2 = {}
for w in words:
    counts2[w] = counts2.get(w, 0) + 1

# 3) setdefault
counts3 = {}
for w in words:
    counts3.setdefault(w, 0)
    counts3[w] += 1

# 4) defaultdict — the one to reach for
counts4 = defaultdict(int)
for w in words:
    counts4[w] += 1

assert dict(counts) == counts2 == counts3 == dict(counts4) == {"ml": 3, "sql": 2, "dl": 1}
print("all four agree:", dict(counts4))

all four agree: {'ml': 3, 'sql': 2, 'dl': 1}


## 8. The GIL — What It Actually Blocks

**Q: Can Python threads speed up a CPU-bound loop? No — why?**

The GIL (Global Interpreter Lock) lets only one thread execute Python bytecode at a time. Threads help **I/O-bound** work (the lock is released while waiting on sockets/disk). For **CPU-bound** work, use `multiprocessing` (separate interpreters, separate GILs) or native extensions (NumPy releases the GIL in its inner loops). Note: **Python 3.13+ ships an optional free-threaded build (PEP 703)** — worth mentioning as a 2026-current answer, but the default CPython still has the GIL.

In [10]:
import time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def cpu_chunk(n):                     # pure-Python CPU work
    return sum(i * i for i in range(n))

N = 3_000_000
serial = time.perf_counter()
a, b = cpu_chunk(N), cpu_chunk(N)
t_serial = time.perf_counter() - serial

with ThreadPoolExecutor(2) as ex:     # GIL blocks real parallelism
    t_thread = time.perf_counter()
    list(ex.map(cpu_chunk, [N, N]))
    t_thread = time.perf_counter() - t_thread

print(f"serial   : {t_serial:.2f}s")
print(f"threads  : {t_thread:.2f}s  <- no faster (GIL); often slower from switching overhead")

serial   : 0.39s
threads  : 0.43s  <- no faster (GIL); often slower from switching overhead


## 9. Everything Is an Object — `type`, `id`, and Mutating Classes

**Predict:** can you attach a new attribute to a *class* at runtime and see it on existing instances?

In [11]:
class Model: pass

m1 = Model()
Model.version = "1.0"               # patch the class itself
print("instance sees it:", m1.version)

m2 = Model()
m1.owner = "me"                     # instance attribute on m1 only
print("m2 has owner?    :", hasattr(m2, "owner"))
print("lookup order     : instance __dict__ first, then class")

instance sees it: 1.0
m2 has owner?    : False
lookup order     : instance __dict__ first, then class


**Why interviewers care:** this is the mechanism behind monkey-patching, class-level config, and — the classic gotcha — **class attributes shared across instances** (a mutable class attribute like `history = []` on an ML estimator is a bug every senior reviewer has caught).

## 10. Hashability — Why Lists Can't Be Dict Keys

**Predict:** does this raise?

In [12]:
try:
    d = {[1, 2]: "value"}
except TypeError as e:
    print("TypeError:", e)

# frozenset works — immutable, therefore hashable
d = {frozenset({1, 2}): "ok"}
print("frozenset key:", d[frozenset({1, 2})])

# tuple of immutables works; tuple containing a list does not
d2 = {(1, 2): "ok"}
print("tuple key    :", d2[(1, 2)])
try:
    d3 = {(1, [2]): "no"}
except TypeError as e:
    print("nested tuple TypeError:", e)

TypeError: unhashable type: 'list'
frozenset key: ok
tuple key    : ok
nested tuple TypeError: unhashable type: 'list'


**The rule:** dict/set keys must be *hashable* — hash stable over lifetime and `__eq__` consistent. Mutable containers fail by design: if a key could change after insertion, the bucket lookup would break. `tuple` yes, `list` no; `frozenset` yes, `set` no.

## 11. `sorted` Stability and Multi-Key Sorts

**Q: Sort users by score descending, then name ascending — one line.** Stability (equal elements keep order) is the property that makes chained sorts work — worth naming explicitly.

In [13]:
users = [("ann", 88), ("bob", 91), ("cat", 88), ("dan", 95), ("eve", 91)]
result = sorted(users, key=lambda u: (-u[1], u[0]))
print(result)
assert result == [("dan", 95), ("bob", 91), ("eve", 91), ("ann", 88), ("cat", 88)]
print("stable: ties keep their relative (name) order — guaranteed by the language spec")

[('dan', 95), ('bob', 91), ('eve', 91), ('ann', 88), ('cat', 88)]
stable: ties keep their relative (name) order — guaranteed by the language spec


## 12. The Generator vs List Comprehension Question

**Q: When do you use `(x for x in ...)` instead of `[x for x in ...]`?**

Generators are lazy and keep **one item in memory** — right for streaming large files or infinite sequences; wrong when you need `len()`, indexing, or two passes. Demonstrating the memory difference with `sys.getsizeof` is the bonus that lands:

In [14]:
import sys
big_list = [i * i for i in range(100_000)]
big_gen  = (i * i for i in range(100_000))
print(f"list comprehension : {sys.getsizeof(big_list):,} bytes for 100,000 ready values")
print(f"generator          : {sys.getsizeof(big_gen):,} bytes — values made on demand")

total = sum(x for x in big_gen)     # generator consumed once
print("sum over generator:", total)
try:
    next(big_gen)
except StopIteration:
    print("generator exhausted — one pass only, by design")

list comprehension : 800,984 bytes for 100,000 ready values
generator          : 200 bytes — values made on demand
sum over generator: 333328333350000
generator exhausted — one pass only, by design
